# LeetCode 3337 — Total Characters in String After Transformations II bằng Python

- Link: https://leetcode.com/problems/total-characters-in-string-after-transformations-ii/
- Độ khó: **Hard**
- Chủ đề: **Đếm theo loại ký tự** (vector 26 chiều), phép biến đổi **tuyến tính** ⇒ **lũy thừa ma trận** `26 × 26` trong `O(26³ log t)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lengthAfterTransformations(self, s: str, t: int, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra độ dài chuỗi chỉ phụ thuộc **số lượng mỗi ký tự**, không phụ thuộc thứ tự ⇒ trạng thái là vector `cnt[0..25]`.
2. Một lần biến đổi là phép **tuyến tính**: `cnt' = M · cnt` với `M[j][i]` = số lần ký tự `i` sinh ra ký tự `j`.
3. `t` lên tới `10^9` ⇒ tính `M^t` bằng **lũy thừa nhị phân** (`≈ 30` phép bình phương).
4. Phân biệt **hướng** của ma trận (cột = nguồn, hàng = đích) — sai hướng cho đáp án sai.
5. Kiểm chứng bằng mô phỏng trực tiếp với `t` nhỏ.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, số lần biến đổi `t`, mảng `nums` độ dài 26. Mỗi lần biến đổi, **mọi** ký tự `c` được thay bằng `nums[c]` ký tự **liền sau** nó trong bảng chữ cái (vòng quanh sau `'z'` về `'a'`). Trả về độ dài chuỗi sau đúng `t` lần, modulo `10^9 + 7`.

**Ví dụ 1:** `s = "abcyy"`, `t = 2`, `nums = [1]*25 + [2]` → `7` (`"bcdzz"` → `"cdeabab"`).
**Ví dụ 2:** `s = "azbk"`, `t = 1`, `nums = [2]*26` → `8`.

**Ràng buộc:** `1 ≤ len(s) ≤ 10^5`; `1 ≤ t ≤ 10^9`; `1 ≤ nums[i] ≤ 25`.

In [1]:
from typing import List
MOD = 10 ** 9 + 7

examples = [("abcyy", 2, [1] * 25 + [2], 7), ("azbk", 1, [2] * 26, 8)]
for s, t, nums, exp in examples:
    print(repr(s), 't =', t, '-> kỳ vọng', exp)

'abcyy' t = 2 -> kỳ vọng 7
'azbk' t = 1 -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1. Ma trận chuyển

`M[j][i] = 1` nếu ký tự `i` sinh ra ký tự `j` trong một bước, tức `j = (i + d) mod 26` với `1 ≤ d ≤ nums[i]`. Vì `nums[i] ≤ 25 < 26`, mỗi ký tự đích xuất hiện tối đa một lần ⇒ các phần tử là `0/1`. **Tổng cột `i` = `nums[i]`**.

In [2]:
def build_matrix(nums):
    M = [[0] * 26 for _ in range(26)]
    for i in range(26):
        for d in range(1, nums[i] + 1):
            M[(i + d) % 26][i] += 1
    return M

M = build_matrix([1] * 25 + [2])
print('cột của z (25):', [j for j in range(26) if M[j][25]], '-> a, b')
print('tổng cột:', [sum(M[j][i] for j in range(26)) for i in (0, 24, 25)])

cột của z (25): [0, 1] -> a, b
tổng cột: [1, 1, 2]


### 3.2. Nhân ma trận modulo & lũy thừa nhị phân

`M^t = ∏ M^{2^b}` trên các bit `b` của `t`. Mỗi phép nhân `26 × 26` tốn `26³ ≈ 1.8·10^4` phép nhân — với `≈ 60` phép nhân là khoảng `10^6`.

In [3]:
def mat_mul(A, B):
    n = len(A)
    C = [[0] * n for _ in range(n)]
    for i in range(n):
        Ai, Ci = A[i], C[i]
        for k in range(n):
            a = Ai[k]
            if a:
                Bk = B[k]
                for j in range(n):
                    Ci[j] += a * Bk[j]
        for j in range(n):
            Ci[j] %= MOD
    return C

def mat_pow(M, e):
    n = len(M)
    R = [[int(i == j) for j in range(n)] for i in range(n)]
    while e:
        if e & 1:
            R = mat_mul(R, M)
        M = mat_mul(M, M)
        e >>= 1
    return R

F = mat_pow([[1, 1], [1, 0]], 10)
print('kiểm tra bằng Fibonacci: F(10) =', F[0][1])

kiểm tra bằng Fibonacci: F(10) = 55


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `cnt = {a:1, b:1, c:1, y:2}`.

- Bước 1: mỗi ký tự sinh đúng 1 ký tự (trừ `z` không có mặt) ⇒ `{b:1, c:1, d:1, z:2}`, độ dài 5.
- Bước 2: `z` sinh `a, b` ⇒ `{c:1, d:1, e:1, a:2, b:2}`, độ dài 7 ✓.

Độ dài = tổng vector đếm. Không cần biết chuỗi trông ra sao.

## 5. Cách 1 — Brute Force: mô phỏng chuỗi

In [4]:
def simulate(s, t, nums):
    for _ in range(t):
        s = ''.join(''.join(chr(97 + (ord(c) - 97 + d) % 26) for d in range(1, nums[ord(c) - 97] + 1)) for c in s)
    return len(s) % MOD

for s, t, nums, exp in examples:
    print(simulate(s, t, nums), '| kỳ vọng', exp)

7 | kỳ vọng 7
8 | kỳ vọng 8


**Độ phức tạp:** độ dài tăng theo cấp số nhân (tới `25^t`) — chỉ dùng cho `t` rất nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

1. `cnt[i]` = số ký tự `i` trong `s`.
2. `M` như mục 3.1; `cnt_t = M^t · cnt`.
3. Đáp án `= Σ_j cnt_t[j] mod p`.

Tương đương: gọi `w = 1ᵀ · M^t` (vector hàng, `w[i]` = độ dài sinh ra từ **một** ký tự `i` sau `t` bước) ⇒ đáp án `= Σ_i w[i] · cnt[i]`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP theo từng bước trên vector đếm (`O(26 · 25 · t)`)

In [5]:
def step_dp(s, t, nums):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    for _ in range(t):
        nxt = [0] * 26
        for i in range(26):
            if cnt[i]:
                for d in range(1, nums[i] + 1):
                    j = (i + d) % 26
                    nxt[j] = (nxt[j] + cnt[i]) % MOD
        cnt = nxt
    return sum(cnt) % MOD

for s, t, nums, exp in examples:
    print(step_dp(s, t, nums), '| kỳ vọng', exp)

7 | kỳ vọng 7
8 | kỳ vọng 8


Tốt cho `t ≤ 10^4`, quá chậm khi `t = 10^9`.

### 7.2. Cách 3 — Lũy thừa ma trận (lời giải chính)

In [6]:
def matrix_power_solution(s, t, nums):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    P = mat_pow(build_matrix(nums), t)
    return sum(P[j][i] * cnt[i] for j in range(26) for i in range(26)) % MOD

for s, t, nums, exp in examples:
    print(matrix_power_solution(s, t, nums), '| kỳ vọng', exp)

7 | kỳ vọng 7
8 | kỳ vọng 8


### 7.3. Cách 4 — Áp lũy thừa trực tiếp lên vector

Không cần tích lũy `R = M^t` dưới dạng ma trận: giữ vector `v` (ban đầu `cnt`), khi bit `b` của `t` bằng 1 thì `v ← M^{2^b} · v` (chỉ `26²` phép), vẫn bình phương `M` mỗi vòng. Số phép nhân ma trận–ma trận giảm còn `⌊log₂ t⌋` (chỉ các bình phương).

In [7]:
def vector_power_solution(s, t, nums):
    v = [0] * 26
    for ch in s:
        v[ord(ch) - 97] += 1
    M = build_matrix(nums)
    while t:
        if t & 1:
            v = [sum(M[j][i] * v[i] for i in range(26)) % MOD for j in range(26)]
        t >>= 1
        if t:
            M = mat_mul(M, M)
    return sum(v) % MOD

for s, t, nums, exp in examples:
    print(vector_power_solution(s, t, nums), '| kỳ vọng', exp)

7 | kỳ vọng 7
8 | kỳ vọng 8


Thứ tự áp `M^{2^b}` không quan trọng vì mọi lũy thừa của cùng một ma trận **giao hoán** với nhau.

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 1 (in vector đếm sau mỗi bước, chỉ các ký tự khác 0).

In [8]:
def trace(s, t, nums):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    show = lambda c: {chr(97 + i): c[i] for i in range(26) if c[i]}
    print('ban đầu:', show(cnt), '| độ dài', sum(cnt))
    for step in range(1, t + 1):
        nxt = [0] * 26
        for i in range(26):
            for d in range(1, nums[i] + 1):
                nxt[(i + d) % 26] += cnt[i]
        cnt = nxt
        print(f'sau bước {step}:', show(cnt), '| độ dài', sum(cnt))
    print('ma trận:', matrix_power_solution(s, t, nums), '| mô phỏng:', simulate(s, t, nums))

trace("abcyy", 3, [1] * 25 + [2])

ban đầu: {'a': 1, 'b': 1, 'c': 1, 'y': 2} | độ dài 5
sau bước 1: {'b': 1, 'c': 1, 'd': 1, 'z': 2} | độ dài 5
sau bước 2: {'a': 2, 'b': 2, 'c': 1, 'd': 1, 'e': 1} | độ dài 7
sau bước 3: {'b': 2, 'c': 2, 'd': 1, 'e': 1, 'f': 1} | độ dài 7
ma trận: 7 | mô phỏng: 7


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def lengthAfterTransformations(self, s: str, t: int, nums: List[int]) -> int:
        MOD = 10 ** 9 + 7
        N = 26

        def mul(A, B):
            C = [[0] * N for _ in range(N)]
            for i in range(N):
                Ai, Ci = A[i], C[i]
                for k in range(N):
                    a = Ai[k]
                    if a:
                        Bk = B[k]
                        for j in range(N):
                            Ci[j] += a * Bk[j]
                for j in range(N):
                    Ci[j] %= MOD
            return C

        M = [[0] * N for _ in range(N)]
        for i in range(N):
            for d in range(1, nums[i] + 1):
                M[(i + d) % N][i] += 1
        v = [0] * N
        for ch in s:
            v[ord(ch) - 97] += 1
        while t:
            if t & 1:
                v = [sum(M[j][i] * v[i] for i in range(N)) % MOD for j in range(N)]
            t >>= 1
            if t:
                M = mul(M, M)
        return sum(v) % MOD

for s, t, nums, exp in examples:
    got = Solution().lengthAfterTransformations(s, t, nums)
    print(got, 'OK' if got == exp else 'SAI')

7 OK
8 OK


## 10. Vì sao thuật toán đúng

1. **Độc lập theo ký tự:** mỗi ký tự biến đổi riêng rẽ, nên chuỗi sau một bước là ghép các khối do từng ký tự sinh ra ⇒ vector đếm mới là tổng có trọng số của các cột `M[:, i]` theo `cnt[i]` ⇒ `cnt' = M · cnt`.
2. **Quy nạp:** sau `t` bước `cnt_t = M^t · cnt_0`.
3. **Độ dài** là tổng các thành phần của `cnt_t`.
4. **Modulo** tương thích với phép cộng và nhân, nên lấy modulo trong mọi phép nhân ma trận vẫn cho đúng `M^t mod p`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(25^t · |s|)` | như trên |
| 2. DP từng bước | `O(26 · 25 · t)` | `O(26)` |
| 3. Lũy thừa ma trận | `O(|s| + 26³ · 2 log t)` | `O(26²)` |
| 4. Lũy thừa lên vector | `O(|s| + 26³ log t + 26² log t)` | `O(26²)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'step_dp': step_dp, 'matrix_power': matrix_power_solution, 'vector_power': vector_power_solution,
         'submit': lambda s, t, nums: Solution().lengthAfterTransformations(s, t, nums)}
edge = [
    ("a", 1, [1] * 26, 1),
    ("z", 1, [25] * 26, 25),
    ("z", 2, [1] * 25 + [2], 2),
    ("a", 26, [1] * 26, 1),
    ("ab", 2, [2] * 26, 8),
]
for s, t, nums, exp in examples + edge:
    assert simulate(s, t, nums) == exp, (s, t, simulate(s, t, nums))
    for name, f in impls.items():
        assert f(s, t, nums) == exp, (name, s, t)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [11]:
import random
rng = random.Random(3337)
cnt = 0
for _ in range(150):
    s = ''.join(rng.choice('abcxyz') for _ in range(rng.randint(1, 4)))
    t = rng.randint(1, 4)
    nums = [rng.randint(1, 3) for _ in range(26)]
    exp = simulate(s, t, nums)
    for name, f in impls.items():
        assert f(s, t, nums) == exp, (name, s, t, nums)
    cnt += 1
for _ in range(100):                      # t lớn hơn: so ma trận với DP từng bước
    s = ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(rng.randint(1, 20)))
    t = rng.randint(1, 300)
    nums = [rng.randint(1, 25) for _ in range(26)]
    assert Solution().lengthAfterTransformations(s, t, nums) == step_dp(s, t, nums) == matrix_power_solution(s, t, nums)
    cnt += 1
print('Khớp trên', cnt, 'bộ test ngẫu nhiên.')

Khớp trên 250 bộ test ngẫu nhiên.


### 12.3. Ca lớn `|s| = 10^5`, `t = 10^9`

In [12]:
import time
big = {
    'nums ngẫu nhiên': (''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(10 ** 5)), 10 ** 9,
                        [rng.randint(1, 25) for _ in range(26)]),
    'nums toàn 1': ('a' * 10 ** 5, 10 ** 9, [1] * 26),
    'nums toàn 25': ('z' * 10 ** 5, 10 ** 9 - 1, [25] * 26),
}
for name, (s, t, nums) in big.items():
    tt = time.time(); a = Solution().lengthAfterTransformations(s, t, nums); t1 = time.time() - tt
    tt = time.time(); b = matrix_power_solution(s, t, nums); t2 = time.time() - tt
    assert a == b
    print(f"{name:16s}: {a:10d} | nộp {t1:.2f}s | ma trận đầy đủ {t2:.2f}s")

nums ngẫu nhiên :  429560697 | nộp 0.06s | ma trận đầy đủ 0.07s
nums toàn 1     :     100000 | nộp 0.01s | ma trận đầy đủ 0.01s


nums toàn 25    :  428569091 | nộp 0.04s | ma trận đầy đủ 0.06s


Kiểm tra độc lập: `nums` toàn 1 ⇒ độ dài không đổi (`10^5`); `nums` toàn `c` ⇒ mỗi ký tự sinh `c` ký tự ⇒ độ dài `|s| · c^t`.

In [13]:
assert Solution().lengthAfterTransformations('a' * 10 ** 5, 10 ** 9, [1] * 26) == 10 ** 5
assert Solution().lengthAfterTransformations('z' * 10 ** 5, 10 ** 9 - 1, [25] * 26) == 10 ** 5 * pow(25, 10 ** 9 - 1, MOD) % MOD
assert Solution().lengthAfterTransformations('abc', 12345, [7] * 26) == 3 * pow(7, 12345, MOD) % MOD
print('nums hằng: khớp công thức |s|·c^t.')

nums hằng: khớp công thức |s|·c^t.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên vòng quanh sau `'z'`

In [14]:
def wrong_no_wrap(s, t, nums):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    for _ in range(t):
        nxt = [0] * 26
        for i in range(26):
            for j in range(i + 1, min(26, i + 1 + nums[i])):     # SAI: bỏ các ký tự vượt 'z'
                nxt[j] += cnt[i]
        cnt = nxt
    return sum(cnt) % MOD

print('sai :', wrong_no_wrap(*examples[0][:3]), '| đúng:', Solution().lengthAfterTransformations(*examples[0][:3]))

sai : 3 | đúng: 7


**Vì sao sai:** `'z'` với `nums[25] = 2` phải sinh `"ab"`; bản sai coi như `'z'` biến mất.

### Lỗi 2 — Dựng ma trận sai hướng (dùng `Mᵀ`)

In [15]:
def wrong_transposed(s, t, nums):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    M = build_matrix(nums)
    MT = [list(r) for r in zip(*M)]                              # SAI: hàng = nguồn
    P = mat_pow(MT, t)
    return sum(P[j][i] * cnt[i] for j in range(26) for i in range(26)) % MOD

nums2 = [3] + [1] * 25
print('sai :', wrong_transposed("a", 1, nums2), '| đúng:', Solution().lengthAfterTransformations("a", 1, nums2))

sai : 1 | đúng: 3


**Vì sao sai:** với `Mᵀ`, tổng cột của `'a'` thành "số ký tự **sinh ra** `'a'`" (chỉ `'z'`, = 1) thay vì "số ký tự `'a'` sinh ra" (= 3). Quy ước: **cột = nguồn, hàng = đích** khi nhân `M · cnt`.

### Lỗi 3 — Giả sử độ dài sinh ra từ `c` là `nums[c]^t` (quên ký tự thay đổi sau mỗi bước)

In [16]:
def wrong_power_of_own(s, t, nums):
    return sum(pow(nums[ord(c) - 97], t, MOD) for c in s) % MOD    # SAI

print('sai :', wrong_power_of_own(*examples[0][:3]), '| đúng:', Solution().lengthAfterTransformations(*examples[0][:3]))

sai : 5 | đúng: 7


**Vì sao sai:** `'y'` thành `'z'` sau bước 1, và ở bước 2 dùng `nums['z'] = 2`, không phải `nums['y'] = 1`.

### Lỗi 4 — Lũy thừa nhị phân lệch một bước (dùng `M^{t−1}` hoặc bắt đầu `R = M`)

In [17]:
def wrong_off_by_one(s, t, nums):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    M = build_matrix(nums)
    R, e, B = M, t, M                                            # SAI: R khởi tạo bằng M (đã thêm 1 bước)
    while e:
        if e & 1:
            R = mat_mul(R, B)
        B = mat_mul(B, B)
        e >>= 1
    return sum(R[j][i] * cnt[i] for j in range(26) for i in range(26)) % MOD

print('sai :', wrong_off_by_one(*examples[1][:3]), '| đúng:', Solution().lengthAfterTransformations(*examples[1][:3]))

sai : 16 | đúng: 8


**Vì sao sai:** kết quả là `M^{t+1}`; với ví dụ 2 ra `16` thay vì `8`. Khởi tạo `R` phải là **ma trận đơn vị**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Độ dài/số lượng sau `t` bước, mỗi phần tử biến đổi độc lập, `t` khổng lồ" ⇒ vector đếm + lũy thừa ma trận.
- Bảng chữ cái 26 ⇒ ma trận nhỏ, `26³ log t` thoải mái.
- **LC 3335 (bản I):** mỗi ký tự sinh đúng 1 ký tự trừ `'z' → "ab"`, `t ≤ 10^5` ⇒ DP từng bước là đủ.

## 15. Biến thể của bài

1. **LC 3335:** quy tắc cố định, `t ≤ 10^5`.
2. Hỏi **số lượng ký tự `'a'`** sau `t` bước ⇒ chỉ lấy thành phần `cnt_t[0]`.
3. Mỗi ký tự có **tập đích tùy ý** (không nhất thiết liên tiếp) ⇒ cùng cách, chỉ đổi `M`.
4. Nhiều truy vấn `t` khác nhau ⇒ lưu sẵn `M^{2^b}`, mỗi truy vấn `O(26² log t)` (cách 4).

In [18]:
# Biến thể 1 (LC 3335) và 4 (nhiều truy vấn t)
def lc3335(s, t):
    return Solution().lengthAfterTransformations(s, t, [1] * 25 + [2])

def lc3335_dp(s, t):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    for _ in range(t):
        z = cnt[25]
        cnt = [z] + cnt[:25]
        cnt[1] = (cnt[1] + z) % MOD
    return sum(cnt) % MOD

assert lc3335("abcyy", 2) == lc3335_dp("abcyy", 2) == 7
assert lc3335("azbk", 1) == lc3335_dp("azbk", 1) == 5

def many_queries(s, nums, ts):
    pw = [build_matrix(nums)]
    for _ in range(max(ts).bit_length()):
        pw.append(mat_mul(pw[-1], pw[-1]))
    base = [0] * 26
    for ch in s:
        base[ord(ch) - 97] += 1
    out = []
    for t in ts:
        v, b = base[:], 0
        while t:
            if t & 1:
                v = [sum(pw[b][j][i] * v[i] for i in range(26)) % MOD for j in range(26)]
            t >>= 1; b += 1
        out.append(sum(v) % MOD)
    return out

nums = [rng.randint(1, 25) for _ in range(26)]
ts = [1, 7, 100, 12345, 10 ** 9]
assert many_queries("leetcode", nums, ts) == [Solution().lengthAfterTransformations("leetcode", t, nums) for t in ts]
print('LC 3335 khớp DP; nhiều truy vấn t khớp từng lời giải riêng.')

LC 3335 khớp DP; nhiều truy vấn t khớp từng lời giải riêng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mô phỏng | mũ | Oracle |
| 2 | DP từng bước | `O(650 t)` | `t` nhỏ |
| 3 | `M^t` đầy đủ | `O(26³ · 2 log t)` | Cổ điển |
| 4 | Lũy thừa lên vector | `O(26³ log t)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "b"`, `t = 3`, `nums = [2] * 26`.

**Bài 2 (tính tay):** `s = "y"`, `t = 2`, `nums = [1]*24 + [2, 3]` (`nums['y'] = 2`, `nums['z'] = 3`).

**Bài 3 (điền TODO):** trả về **vector đếm** 26 thành phần sau `t` bước.

**Bài 4 (tự viết lại):** tìm `t` **nhỏ nhất** để độ dài (không modulo) vượt ngưỡng `L` (`L ≤ 10^18`).

**Bài 5 (mini challenge):** biến thể 3 — mỗi ký tự `i` có danh sách đích `targets[i]` tùy ý.

In [19]:
def count_vector_todo(s, t, nums):
    # TODO
    raise NotImplementedError

def first_exceed_todo(s, nums, L):
    # TODO
    raise NotImplementedError

def arbitrary_targets_todo(s, t, targets):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mỗi ký tự sinh 2. (2) `2^3`. (3) `8`.

**Bài 2.** (1) Bước 1: `y → z, a`. (2) Bước 2: `z → a, b, c`; `a → b`. (3) `4`.

**Bài 3.** (1) Cách 4 nhưng trả `v`. (2) Modulo từng thành phần. (3) So với DP từng bước.

**Bài 4.** (1) Độ dài không giảm (mỗi ký tự sinh ≥ 1). (2) Nếu mọi `nums[c] = 1` cho các ký tự **có thể xuất hiện** thì không bao giờ vượt. (3) Ngược lại mọi ký tự đi vòng qua đủ 26 chữ cái trong ≤ 26 bước, nên độ dài tăng ít nhất 1 sau mỗi 26 bước ⇒ mô phỏng vector với giá trị **bão hòa** ở `L + 1` (hoặc nhị phân trên `t` với lũy thừa ma trận bão hòa khi `L` lớn).

**Bài 5.** (1) `M[j][i] += 1` cho mỗi `j ∈ targets[i]`. (2) Giữ nguyên phần còn lại. (3) Mô phỏng để kiểm.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().lengthAfterTransformations("b", 3, [2] * 26) == 8
assert Solution().lengthAfterTransformations("y", 2, [1] * 24 + [2, 3]) == 4 == simulate("y", 2, [1] * 24 + [2, 3])

def count_vector(s, t, nums):
    v = [0] * 26
    for ch in s:
        v[ord(ch) - 97] += 1
    M = build_matrix(nums)
    while t:
        if t & 1:
            v = [sum(M[j][i] * v[i] for i in range(26)) % MOD for j in range(26)]
        t >>= 1
        if t:
            M = mat_mul(M, M)
    return v

rng = random.Random(3)
for _ in range(50):
    s = ''.join(rng.choice('abxyz') for _ in range(rng.randint(1, 5)))
    t = rng.randint(1, 50); nums = [rng.randint(1, 25) for _ in range(26)]
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    for _ in range(t):
        nxt = [0] * 26
        for i in range(26):
            for d in range(1, nums[i] + 1):
                nxt[(i + d) % 26] = (nxt[(i + d) % 26] + cnt[i]) % MOD
        cnt = nxt
    assert count_vector(s, t, nums) == cnt
print('Bài 1: 8 | Bài 2: 4 | Bài 3 khớp DP từng bước.')

Bài 1: 8 | Bài 2: 4 | Bài 3 khớp DP từng bước.


In [21]:
# Bài 4, 5
def first_exceed(s, nums, L, t_max=10 ** 6):
    """t nhỏ nhất để độ dài > L (mô phỏng vector với chặn bão hòa), None nếu không vượt trong t_max bước."""
    cap = L + 1
    v = [0] * 26
    for ch in s:
        v[ord(ch) - 97] += 1
    if sum(v) > L:
        return 0
    if all(x == 1 for x in nums):          # độ dài không bao giờ đổi
        return None
    for t in range(1, t_max + 1):
        nxt = [0] * 26
        for i in range(26):
            if v[i]:
                for d in range(1, nums[i] + 1):
                    j = (i + d) % 26
                    nxt[j] = min(cap, nxt[j] + v[i])
        v = nxt
        if sum(v) > L:
            return t
    return None

def arbitrary_targets(s, t, targets):
    M = [[0] * 26 for _ in range(26)]
    for i in range(26):
        for j in targets[i]:
            M[j][i] += 1
    v = [0] * 26
    for ch in s:
        v[ord(ch) - 97] += 1
    P = mat_pow(M, t)
    return sum(P[j][i] * v[i] for j in range(26) for i in range(26)) % MOD

rng = random.Random(4)
for _ in range(60):
    s = ''.join(rng.choice('abyz') for _ in range(rng.randint(1, 3)))
    nums = [rng.randint(1, 3) for _ in range(26)]
    L = rng.randint(1, 200)
    t0 = first_exceed(s, nums, L)
    assert t0 is not None and simulate(s, t0, nums) > L and (t0 == 0 or simulate(s, t0 - 1, nums) <= L)
    targets = [rng.sample(range(26), rng.randint(1, 3)) for _ in range(26)]
    exp = s
    for _ in range(3):
        exp = ''.join(chr(97 + j) for c in exp for j in targets[ord(c) - 97])
    assert arbitrary_targets(s, 3, targets) == len(exp) % MOD
assert first_exceed("abc", [1] * 26, 10, t_max=100) is None
print('Bài 4, 5 khớp mô phỏng.')

Bài 4, 5 khớp mô phỏng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
M[(i + d) % 26][i] += 1   for d in 1..nums[i]       # cột = nguồn, hàng = đích
v = cnt(s)
while t: if t & 1: v = M·v;  t >>= 1;  M = M·M
return sum(v) % MOD
```

**Checklist:**

- [ ] Vòng quanh `% 26`?
- [ ] Cột là nguồn, hàng là đích (nhân `M · v`)?
- [ ] Khởi tạo lũy thừa bằng ma trận đơn vị (hoặc vector gốc)?
- [ ] Modulo trong mọi phép nhân ma trận?
- [ ] Đếm theo loại ký tự, không mô phỏng chuỗi?